In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

In [ ]:
df = pd.read_csv("../data/raw/all_crypto_currencies.csv")
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print("Dataset shape:", df.shape)


In [ ]:
print(df.columns)
print(df.head())

In [ ]:
print("\n=== Missing Values ===")
missing_counts = df.isnull().sum()
missing_percent = (missing_counts / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing %': missing_percent
}).sort_values(by='Missing Count', ascending=False)

print(missing_df[missing_df['Missing Count'] > 0])

print("\n=== Duplicate Rows ===")
duplicate_count = df.duplicated().sum()
print(f"Total duplicate rows: {duplicate_count}")

# Optional: view them
duplicates = df[df.duplicated()]
print(duplicates.head())

print("\n=== Duplicate (date + slug) ===")
dup_key = df.duplicated(subset=['date', 'slug']).sum()
print(f"Duplicate (date, symbol) rows: {dup_key}")

print("\n=== Empty Strings (Hidden Missing) ===")
empty_strings = (df == "").sum()
print(empty_strings[empty_strings > 0])

print("\n=== Dataset Info ===")
df.info()

print("\n=== Statistical Summary ===")
print(df.describe(include='all'))

In [ ]:
# 1. Correct duplicate check
df.duplicated(subset=['date', 'slug']).sum()

# 2. Symbol collision
df.groupby('symbol')['slug'].nunique().sort_values(ascending=False).head()

# 3. Inspect one duplicate group
dupes = df[df.duplicated(subset=['date', 'symbol'], keep=False)]
dupes.iloc[:20]

In [ ]:
print(df.columns)
print(df.head())

In [ ]:
# =============================================================================
# STEP 4 — CHRONOLOGICAL SPLIT (date-proportional across all coins)
#
# Splitting by date (not row index) ensures every coin's data is represented
# in each fold, and no future dates appear in earlier folds.
# =============================================================================
 
df = df.sort_values("date").reset_index(drop=True)
 
total = len(df)
date_counts = df.groupby("date").size().reset_index(name="count")
date_counts["cum"] = date_counts["count"].cumsum()
date_counts["pct"] = date_counts["cum"] / total
 
train_end = date_counts.loc[date_counts["pct"] >= 0.70, "date"].iloc[0]
val_end   = date_counts.loc[date_counts["pct"] >= 0.85, "date"].iloc[0]
 
train_df = df[df["date"] <= train_end].copy()
val_df   = df[(df["date"] > train_end) & (df["date"] <= val_end)].copy()
test_df  = df[df["date"] > val_end].copy()
 
print(f"\nSplit: Train={len(train_df):,} | Val={len(val_df):,} | Test={len(test_df):,}")
print(f"  Train dates: up to {train_end.date()}")
print(f"  Val dates:   up to {val_end.date()}")
print(f"  Test dates:  after {val_end.date()}")


In [ ]:
import pandas as pd
import numpy as np

# Sort dataset
df = df.sort_values(['slug', 'date']).reset_index(drop=True)

# Precompute repeated groupings
grouped = df.groupby('slug', group_keys=False)

# Vectorized Features
df['daily_return'] = grouped['close'].pct_change()

# Rolling features
df['ma_7'] = grouped['close'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
df['ma_30'] = grouped['close'].rolling(30, min_periods=1).mean().reset_index(level=0, drop=True)
df['vol_7'] = grouped['daily_return'].rolling(7, min_periods=1).std().reset_index(level=0, drop=True)
df['vol_30'] = grouped['daily_return'].rolling(30, min_periods=1).std().reset_index(level=0, drop=True)
df['rolling_max_7'] = grouped['high'].rolling(7, min_periods=1).max().reset_index(level=0, drop=True)
df['rolling_min_7'] = grouped['low'].rolling(7, min_periods=1).min().reset_index(level=0, drop=True)

# Lag Features
df['lag_1'] = grouped['close'].shift(1)
df['lag_7'] = grouped['close'].shift(7)

# Momentum
df['momentum_7'] = df['close'] - df['lag_7']
df['momentum_14'] = df['close'] - grouped['close'].shift(14)

# Volume
df['vol_ma_7'] = grouped['volume'].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
df['vol_ratio'] = df['volume'] / (df['vol_ma_7'] + 1e-9)

# Price spreads
df['high_low_spread'] = df['high'] - df['low']
df['close_open_spread'] = df['close'] - df['open']
df['high_close_ratio'] = df['high'] / (df['close'] + 1e-9)

# Market features
df['market_cap_ratio'] = df['market'] / df.groupby('date')['market'].transform('sum')
df['rank_normalized'] = df['ranknow'] / df['ranknow'].max()

# EMA
df['ema_7'] = grouped['close'].transform(lambda x: x.ewm(span=7, adjust=False).mean())
df['ema_14'] = grouped['close'].transform(lambda x: x.ewm(span=14, adjust=False).mean())

# RSI
def rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(period, min_periods=1).mean()
    avg_loss = loss.rolling(period, min_periods=1).mean()
    rs = avg_gain / (avg_loss + 1e-9)
    return 100 - (100 / (1 + rs))

df['rsi_14'] = grouped['close'].transform(rsi)

# MACD
ema_12 = grouped['close'].transform(lambda x: x.ewm(span=12, adjust=False).mean())
ema_26 = grouped['close'].transform(lambda x: x.ewm(span=26, adjust=False).mean())
df['macd'] = ema_12 - ema_26
df['macd_signal'] = grouped['macd'].transform(lambda x: x.ewm(span=9, adjust=False).mean())

# Volatility clustering
df['vol_cluster_7'] = grouped['daily_return'].rolling(7, min_periods=1).std().reset_index(level=0, drop=True)
df['vol_cluster_14'] = grouped['daily_return'].rolling(14, min_periods=1).std().reset_index(level=0, drop=True)

# Handle NaNs
df.fillna(0, inplace=True)

print("Optimized feature engineering done:", df.shape)
# Show all columns in the dataset when printing
import pandas as pd

# Make sure all columns are visible
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Display the first 10 rows to inspect all features
print(df.head(10))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Select numeric features only
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns

# Compute correlation matrix
corr_matrix = df[numeric_cols].corr()

# Display a heatmap
plt.figure(figsize=(16,12))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', cbar=True)
plt.title("Correlation Matrix of Features")
plt.show()

In [ ]:
# Set a correlation threshold
threshold = 0.9

# Find pairs of features with correlation > threshold
high_corr_pairs = []

for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if abs(corr_matrix.iloc[i, j]) > threshold:
            high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i,j]))

# Display highly correlated pairs
for f1, f2, corr_val in high_corr_pairs:
    print(f"{f1} ↔ {f2} | Correlation: {corr_val:.2f}")

In [ ]:
# List of features to drop
drop_features = ['ma_7', 'ema_7', 'vol_cluster_7', 'volume', 'vol_ma_7', 'spread', 'ranknow']

# Drop the features
df = df.drop(columns=drop_features)

# Verify the remaining features
print("Remaining features after dropping redundant ones:")
print(df.columns.tolist())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Select numeric features only
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns

# Compute correlation matrix for remaining features
corr_matrix = df[numeric_cols].corr()

# Display a heatmap
plt.figure(figsize=(16,12))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', cbar=True)
plt.title("Correlation Matrix of Remaining Features")
plt.show()

# Set a correlation threshold
threshold = 0.9

# Find pairs of features with correlation > threshold
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if abs(corr_matrix.iloc[i, j]) > threshold:
            high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i,j]))

# Display highly correlated pairs
print("Highly correlated feature pairs (>|0.9|):")
for f1, f2, corr_val in high_corr_pairs:
    print(f"{f1} ↔ {f2} | Correlation: {corr_val:.2f}")